# CAM equilibrium check

This notebook is for **CAM monthly history files**.

*Author: ChatGPT (22 April 2026)*

In [13]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys
from pathlib import Path

# Add project root /src to path
sys.path.append(str(Path.cwd().parents[1] / "src"))

# Your package
from boreal_forest_expansion.core import (
    open_mfdataset_sorted,
    add_derived_cam_fields,
    reduce_series,
    annual_mean,
)

from boreal_forest_expansion.diagnostics import equilibrium_test, moving_window_trends

In [15]:
path_to_archive='/nird/datapeak/NS9188K/adelez/BRL-FRST-XPSN_archive/NF2000norbc_tropstratchem_spinup_f19_f19/atm/hist/'

In [17]:
CAM_GLOB = path_to_archive+"*.cam.h0.*.nc"
OUTDIR = "./cam_equilibrium_output"

CAM_VARS = ["RESTOM","TREFHT","PRECT","FSNT","FLNT","PSL","CLDTOT"]

# Equilibrium parameters
LAST_N_YEARS = 10
MOVING_WINDOW = 10
ROLLING_WINDOW = 5
REL_DRIFT_THRESHOLD = 0.01
PVAL_THRESHOLD = 0.05

REGIONS = {
    "global": None,
    "tropics": (-23.5, 23.5, -180.0, 180.0),
    "NH_extratropics": (23.5, 90.0, -180.0, 180.0),
    "SH_extratropics": (-90.0, -23.5, -180.0, 180.0),
    "amazon": (-20.0, 10.0, -80.0, -45.0),
    "europe": (35.0, 72.0, -10.0, 40.0),
    "boreal": (50.0, 75.0, -180.0, 180.0),
    "arctic": (66.5, 90.0, -180.0, 180.0),
}

Path(OUTDIR).mkdir(parents=True, exist_ok=True)


## 4. Open CAM dataset

In [19]:
cam_ds = open_mfdataset_sorted(CAM_GLOB)
cam_ds = add_derived_cam_fields(cam_ds)

print("Time span:", str(cam_ds.time.values[0]), "to", str(cam_ds.time.values[-1]))
print(cam_ds)

FileNotFoundError: No files found for pattern: /nird/datapeak/NS9188K/adelez/BRL-FRST-XPSN_archive/NF2000norbc_tropstratchem_spinup_f19_f19/atm/hist/*.cam.h0.*.nc

## 5. Check available variables

In [10]:
cam_available = [v for v in CAM_VARS if v in cam_ds.data_vars]
cam_missing = [v for v in CAM_VARS if v not in cam_ds.data_vars]
print("CAM available:", cam_available)
print("CAM missing:", cam_missing)


NameError: name 'cam_ds' is not defined

## 6. Plot one CAM variable interactively

In [ ]:
def plot_one_cam_variable(variable: str, region_name: str = "global"):
    region_bounds = REGIONS[region_name]
    da = maybe_convert_units(cam_ds[variable], variable)
    series = reduce_series(da, region_bounds)
    ann = annual_mean(series)
    summary, y_last, fitted_last = equilibrium_test(
        ann, last_n_years=LAST_N_YEARS,
        rel_drift_threshold=REL_DRIFT_THRESHOLD,
        pval_threshold=PVAL_THRESHOLD,
    )
    summary.variable = variable
    summary.region = region_name

    years = pd.to_datetime(ann["time"].values).year.astype(float)
    vals = ann.values.astype(float)
    roll = rolling_mean(vals, ROLLING_WINDOW)
    units = get_units(ann)

    plt.figure(figsize=(9, 5.5))
    plt.plot(years, vals, marker="o", linewidth=1.4, label="Annual mean")
    plt.plot(years, roll, linewidth=2.0, label=f"{ROLLING_WINDOW}-yr rolling mean")
    plt.plot(y_last, fitted_last, linestyle="--", linewidth=2.0, label=f"Late-run trend ({LAST_N_YEARS} yr)")
    plt.title(f"CAM | {variable} | {region_name}\nrel. change last {LAST_N_YEARS} yr = {summary.rel_change_last_period:.3%}, slope = {summary.slope_per_decade:.3g} / decade, eq = {summary.equilibrium_flag}")
    plt.xlabel("Year")
    plt.ylabel(f"{variable} ({units})" if units else variable)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    trend_df = moving_window_trends(ann, MOVING_WINDOW)
    if not trend_df.empty:
        plt.figure(figsize=(9, 4.8))
        plt.plot(trend_df["year_end"], trend_df["slope_per_decade"], marker="o", linewidth=1.5)
        plt.axhline(0.0, linestyle="--", linewidth=1.0)
        plt.title(f"CAM | {variable} | {region_name}\nMoving {MOVING_WINDOW}-year trend")
        plt.xlabel("End year of moving window")
        plt.ylabel(f"Slope per decade [{units}/decade]" if units else "Slope per decade")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    return summary, ann, trend_df


## 7. Example CAM plots

In [ ]:
summary_restom, ann_restom, trend_restom = plot_one_cam_variable("RESTOM", "global")
summary_trefht, ann_trefht, trend_trefht = plot_one_cam_variable("TREFHT", "global")
summary_restom, summary_trefht
